In [1]:
import duckdb

# Your real store root (same path run_daily_stats.py uses).
PARSED_ROOT = ("/Users/shazzak/Library/CloudStorage/"
               "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

# All trades partitions: <root>/trades/date=YYYY-MM-DD/*.parquet
trades_glob = f"{PARSED_ROOT}/trades/*/*.parquet"

query = f"""
SELECT
  -- strip the -JUL / -JULB suffix to get the underlying (OGDC-JUL -> OGDC)
  regexp_replace(symbol, '-.*$', '') AS underlying,
  segment,
  count(DISTINCT date)          AS days_traded,
  count(*)                      AS trades,
  round(sum(qty * price) / 1e6, 1) AS notional_m
FROM read_parquet('{trades_glob}', hive_partitioning=true)
WHERE segment IN ('STOCK_DEL_FUT', 'STOCK_CS_FUT')
GROUP BY 1, 2
ORDER BY underlying, notional_m DESC
"""

df = duckdb.sql(query).df()

# Show the whole result in the notebook.
import pandas as pd
pd.set_option("display.max_rows", None, "display.width", 200)
print(f"{len(df)} rows")
df

0 rows


,underlying,segment,days_traded,trades,notional_m


In [ ]:
df.tail()


In [2]:
import duckdb, glob
staged = len(glob.glob("daily_stats_staging/daily_stats_*.parquet"))
n, d, lo, hi = duckdb.sql("""
    SELECT count(*), count(DISTINCT date), min(date), max(date)
    FROM 'daily_stats_ALL.parquet'
""").fetchone()
print(f"staging files: {staged}")
print(f"merged: {n:,} rows | {d} distinct dates | {lo} -> {hi}")
print("MATCH" if staged == d else "MISMATCH -- do not delete")

staging files: 207
merged: 122,912 rows | 207 distinct dates | 2025-09-01 -> 2026-06-30
MATCH


In [3]:
import duckdb
duckdb.sql("""
    SELECT date, count(*) AS symbols, round(sum(notional_m),1) AS notional_m
    FROM 'daily_stats_ALL.parquet'
    GROUP BY date ORDER BY symbols
    LIMIT 10
""").df()

,date,symbols,notional_m
0,2026-03-09,538,44728.0
1,2026-03-06,539,29271.9
2,2026-03-19,542,15184.7
3,2026-04-03,545,19821.4
4,2026-03-12,548,31416.5
5,2026-03-13,549,22872.9
6,2025-12-03,550,57194.1
7,2025-11-07,550,38785.1
8,2026-03-31,552,29663.0
9,2025-12-01,553,43292.4


In [4]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

duckdb.sql(f"""
    SELECT date,
           min(transact_time) AS first_trade,
           max(transact_time) AS last_trade,
           round(date_diff('minute', min(transact_time), max(transact_time)) / 60.0, 2) AS session_hours,
           count(*) AS trades
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    WHERE initiator <> 'AUCTION'
    GROUP BY date
    ORDER BY session_hours
    LIMIT 20
""").df()

,date,first_trade,last_trade,session_hours,trades
0,2026-03-13,2026-03-13 09:17:00.010000+05:00,2026-03-13 12:29:59.940000+05:00,3.20,210486
1,2026-02-20,2026-02-20 09:17:00.100000+05:00,2026-02-20 12:29:59.900000+05:00,3.20,329010
2,2026-02-27,2026-02-27 09:17:00.010000+05:00,2026-02-27 12:29:59.910000+05:00,3.20,386717
3,2026-03-06,2026-03-06 09:17:00.030000+05:00,2026-03-06 12:29:59.920000+05:00,3.20,296241
4,2026-03-19,2026-03-19 09:25:49.160000+05:00,2026-03-19 13:29:59.970000+05:00,4.07,141560
5,2026-03-12,2026-03-12 09:17:00.040000+05:00,2026-03-12 13:29:59.990000+05:00,4.20,267925
6,2026-02-19,2026-02-19 09:17:00.010000+05:00,2026-02-19 13:29:59.980000+05:00,4.20,341845
7,2026-02-24,2026-02-24 09:17:00.010000+05:00,2026-02-24 13:29:59.980000+05:00,4.20,484402
8,2026-03-02,2026-03-02 09:17:00.010000+05:00,2026-03-02 13:29:59.920000+05:00,4.20,370596
9,2026-03-11,2026-03-11 09:17:00.070000+05:00,2026-03-11 13:29:59.880000+05:00,4.20,308129


In [5]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

# Build a session calendar: elapsed span, actual traded minutes, largest intra-day
# gap (a break shows up as a large gap), and day of week.
cal = duckdb.sql(f"""
    WITH t AS (
        SELECT date, transact_time AS ts
        FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE initiator <> 'AUCTION'
    ),
    g AS (
        SELECT date, ts,
               date_diff('second', lag(ts) OVER (PARTITION BY date ORDER BY ts), ts) AS gap_s
        FROM t
    )
    SELECT date,
           dayname(date)                                        AS dow,
           min(ts)                                              AS open_ts,
           max(ts)                                              AS close_ts,
           round(date_diff('second', min(ts), max(ts))/3600.0, 2) AS elapsed_h,
           round(max(gap_s)/60.0, 1)                            AS max_gap_min,
           round((date_diff('second', min(ts), max(ts)) - coalesce(max(gap_s),0))/3600.0, 2) AS traded_h,
           count(*)                                             AS trades
    FROM g
    GROUP BY date
    ORDER BY date
""").df()

# Classify the regime from the measured session, not from a calendar assumption.
cal["regime"] = "normal"
cal.loc[(cal.dow == "Friday"), "regime"] = "friday"
ram = (cal.date >= "2026-02-19") & (cal.date <= "2026-03-19")
cal.loc[ram & (cal.dow != "Friday"), "regime"] = "ramadan"
cal.loc[ram & (cal.dow == "Friday"), "regime"] = "ramadan_friday"

print(cal.groupby("regime")[["elapsed_h", "traded_h", "max_gap_min", "trades"]].median())
cal.to_parquet("session_calendar.parquet", index=False)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                elapsed_h  traded_h  max_gap_min    trades
regime                                                    
friday               7.22      4.68        152.0  483524.5
normal               5.97      5.96          0.5  495123.5
ramadan              4.22      4.21          0.5  341845.0
ramadan_friday       3.22      3.21          0.5  312625.5


In [6]:
import duckdb, pandas as pd
pd.set_option("display.width", 220, "display.max_columns", None)

PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
D   = "2025-09-23"
SYM = "DMC"

SNAP = f"{PARSED}/ob_snapshot/date={D}/*.parquet"
TRD  = f"{PARSED}/trades/date={D}/*.parquet"

# 1. THE DIRECT CONFIRMATION: which entry_types exist, at which levels?
#    If OFFER never appears at level 1, the pivot has no OFFER column -> KeyError.
print("=== 1. entry_type x level census ===")
print(duckdb.sql(f"""
    SELECT entry_type, level, count(*) AS rows,
           count(DISTINCT msg_seq) AS msgs,
           min(px) AS min_px, max(px) AS max_px, sum(qty) AS tot_qty
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
    GROUP BY entry_type, level
    ORDER BY entry_type, level
""").df().to_string(index=False))

# 2. Level 1 only, both sides side by side. A zero OFFER count is the smoking gun.
print("\n=== 2. level-1 BID vs OFFER ===")
print(duckdb.sql(f"""
    SELECT
      count(*) FILTER (WHERE entry_type='BID'   AND level=1) AS bid_l1_rows,
      count(*) FILTER (WHERE entry_type='OFFER' AND level=1) AS offer_l1_rows,
      count(DISTINCT msg_seq) AS total_snapshot_msgs
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
""").df().to_string(index=False))

# 3. Session shape: phase / trading_status over the day, and whether it was suspended.
print("\n=== 3. phase & status ===")
print(duckdb.sql(f"""
    SELECT phase, trading_status, suspended_all_day, break_reason,
           count(DISTINCT msg_seq) AS msgs,
           min(orig_time) AS first_msg, max(orig_time) AS last_msg
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
    GROUP BY phase, trading_status, suspended_all_day, break_reason
    ORDER BY msgs DESC
""").df().to_string(index=False))

# 4. WAS IT LOCKED LIMIT-UP? Compare the best bid against the upper circuit limit.
#    A bid sitting at/above the cap with no offers is the classic one-sided cause.
print("\n=== 4. circuit limits vs best bid ===")
print(duckdb.sql(f"""
    SELECT
      max(px) FILTER (WHERE entry_type='UPPER_CIRCUIT_BREAKER') AS limit_up,
      max(px) FILTER (WHERE entry_type='LOWER_CIRCUIT_BREAKER') AS limit_dn,
      max(px) FILTER (WHERE entry_type='BID'   AND level=1)     AS best_bid,
      max(px) FILTER (WHERE entry_type='OFFER' AND level=1)     AS best_ask,
      max(prev_close)                                           AS prev_close
    FROM read_parquet('{SNAP}')
    WHERE symbol = '{SYM}'
""").df().to_string(index=False))

# 5. The 201 shares: were they auction prints (no continuous aggressor) or continuous?
print("\n=== 5. the trades ===")
print(duckdb.sql(f"""
    SELECT initiator, aggressor_side, count(*) AS n, sum(qty) AS shares,
           min(price) AS min_px, max(price) AS max_px,
           min(transact_time) AS first, max(transact_time) AS last
    FROM read_parquet('{TRD}')
    WHERE symbol = '{SYM}'
    GROUP BY initiator, aggressor_side
    ORDER BY n DESC
""").df().to_string(index=False))

# 6. CONTROL: a healthy symbol the same day, to prove the query itself is sound.
print("\n=== 6. control (OGDC, same day) ===")
print(duckdb.sql(f"""
    SELECT
      count(*) FILTER (WHERE entry_type='BID'   AND level=1) AS bid_l1_rows,
      count(*) FILTER (WHERE entry_type='OFFER' AND level=1) AS offer_l1_rows
    FROM read_parquet('{SNAP}')
    WHERE symbol = 'OGDC'
""").df().to_string(index=False))

=== 1. entry_type x level census ===
           entry_type  level  rows  msgs  min_px  max_px  tot_qty
              AGG_BID      0    19    19   80.67   80.67  20879.0
                  BID      1    19    19   80.67   80.67  20879.0
           LAST_TRADE      0    11    11   80.67   80.67   1205.0
LOWER_CIRCUIT_BREAKER      0    28    28   66.01   66.01      0.0
         NET_CHANGE_1      0    11    11    7.33    7.33      0.0
         NET_CHANGE_2      0    11    11    0.00    7.33      0.0
        OPENING_PRICE      0    11    11   80.67   80.67      0.0
         SESSION_HIGH      0    11    11   80.67   80.67      0.0
          SESSION_LOW      0    11    11   80.67   80.67      0.0
UPPER_CIRCUIT_BREAKER      0    28    28   80.67   80.67      0.0

=== 2. level-1 BID vs OFFER ===
 bid_l1_rows  offer_l1_rows  total_snapshot_msgs
          19              0                   28

=== 3. phase & status ===
             phase trading_status  suspended_all_day   break_reason  msgs      

In [8]:
import duckdb, pandas as pd
pd.set_option("display.width", 200)
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")

print(duckdb.sql(f"""
    WITH traded AS (
        SELECT date, symbol
        FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE symbol IS NOT NULL
        GROUP BY 1, 2
    ),
    screened AS (
        SELECT date, symbol FROM 'daily_stats_ALL.parquet'
    )
    SELECT t.symbol,
           count(*)                     AS days_traded,
           count(s.symbol)              AS days_screened,
           count(*) - count(s.symbol)   AS days_dropped,
           round(100.0 * (count(*) - count(s.symbol)) / count(*), 1) AS pct_dropped
    FROM traded t
    LEFT JOIN screened s ON t.date = s.date AND t.symbol = s.symbol
    GROUP BY t.symbol
    HAVING count(*) >= 20
    ORDER BY pct_dropped DESC
    LIMIT 30
""").df().to_string(index=False))

    symbol  days_traded  days_screened  days_dropped  pct_dropped
   AGLNCPS           51             14            37         72.5
  MCB-DECB           20             10            10         50.0
   GEMNETS          128             70            58         45.3
   GEMBCEM           85             48            37         43.5
  AGP-MAYB           20             12             8         40.0
      ASIC          140             86            54         38.6
    GEMMEL          132             82            50         37.9
   MTL-FEB           20             13             7         35.0
GLAXO-OCTB           23             15             8         34.8
   MTL-JAN           26             17             9         34.6
      EWIC          117             77            40         34.2
   GEMPAPL           88             58            30         34.1
  GEMBLUEX           73             51            22         30.1
  ILP-OCTB           20             14             6         30.0
  BAHL-JAN

In [9]:
print(duckdb.sql(f"""
    WITH traded AS (
        SELECT date, symbol FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE symbol IS NOT NULL GROUP BY 1,2
    ),
    screened AS (SELECT date, symbol FROM 'daily_stats_ALL.parquet'),
    dropped AS (
        SELECT t.date, t.symbol FROM traded t
        LEFT JOIN screened s ON t.date=s.date AND t.symbol=s.symbol
        WHERE s.symbol IS NULL
    )
    SELECT d.date, dayname(d.date) AS dow,
           count(*) AS symbols_dropped,
           (SELECT count(*) FROM traded t2 WHERE t2.date = d.date) AS symbols_traded,
           round(100.0*count(*) / (SELECT count(*) FROM traded t2 WHERE t2.date = d.date), 1) AS pct
    FROM dropped d GROUP BY d.date
    ORDER BY symbols_dropped DESC LIMIT 25
""").df().to_string(index=False))

      date       dow  symbols_dropped  symbols_traded  pct
2026-03-02    Monday               60             618  9.7
2026-02-19  Thursday               49             631  7.8
2026-03-09    Monday               47             585  8.0
2026-03-19  Thursday               42             584  7.2
2026-02-16    Monday               39             601  6.5
2026-03-17   Tuesday               37             600  6.2
2026-02-23    Monday               36             656  5.5
2025-11-21    Friday               36             616  5.8
2026-03-27    Friday               35             647  5.4
2026-02-20    Friday               34             616  5.5
2025-11-19 Wednesday               34             609  5.6
2026-03-13    Friday               33             582  5.7
2026-03-06    Friday               33             572  5.8
2026-04-06    Monday               33             587  5.6
2026-03-10   Tuesday               32             593  5.4
2025-11-27  Thursday               32             672  4

In [10]:
print(duckdb.sql(f"""
    SELECT symbol, market, segment, count(DISTINCT date) AS days, count(*) AS trades
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    WHERE symbol IN ('GEMNETS','GEMBCEM','GEMMEL','AGLNCPS','POWERPS','ANLNV',
                     'ASIC','EWIC','OGDC','KTML')
    GROUP BY symbol, market, segment ORDER BY symbol
""").df().to_string(index=False))

 symbol market segment  days  trades
AGLNCPS    REG     011    51     106
  ANLNV    REG     011    45     421
   ASIC    REG     011   140     974
   EWIC    REG     011   117     559
GEMBCEM    REG     011    85     281
 GEMMEL    REG     011   132     758
GEMNETS    REG     011   128     340
   KTML    REG     011   207   98720
   KTML    NDM     091    18      24
   OGDC    NDM     091    92     189
   OGDC    REG     011   207 1476608
POWERPS    REG     011   187    1689


In [11]:
print(duckdb.sql(f"""
    WITH traded AS (
        SELECT date, symbol FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE symbol IS NOT NULL GROUP BY 1,2
    ),
    screened AS (SELECT date, symbol FROM 'daily_stats_ALL.parquet')
    SELECT t.symbol, count(*) AS days_traded, count(s.symbol) AS days_screened,
           round(100.0*(count(*)-count(s.symbol))/count(*),1) AS pct_dropped
    FROM traded t LEFT JOIN screened s ON t.date=s.date AND t.symbol=s.symbol
    WHERE t.symbol IN ('KTML','FFC','SYS','BAFL','PSO','HBL','KOHC','PKGS',
                       'ITANZ','AKBL','NBP','SGPL','OGDC','MCB')
    GROUP BY t.symbol ORDER BY pct_dropped DESC
""").df().to_string(index=False))

symbol  days_traded  days_screened  pct_dropped
 ITANZ           89             75         15.7
  SGPL          207            203          1.9
   SYS          207            207          0.0
   NBP          207            207          0.0
  KOHC          207            207          0.0
  AKBL          207            207          0.0
  KTML          207            207          0.0
   MCB          207            207          0.0
   HBL          207            207          0.0
  BAFL          207            207          0.0
   PSO          207            207          0.0
   FFC          207            207          0.0
  OGDC          207            207          0.0
  PKGS          207            207          0.0


In [15]:
print(duckdb.sql(f"""
    SELECT min(date) AS first_day, max(date) AS last_day, count(DISTINCT date) AS days
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    WHERE symbol = 'ITANZ'
""").df().to_string(index=False))

 first_day   last_day  days
2026-02-09 2026-06-30    89


In [14]:
# A. Are the dropped days NDM-only?
print(duckdb.sql(f"""
    WITH traded AS (
        SELECT date, symbol,
               count(*) FILTER (WHERE market =  'REG') AS reg_trades,
               count(*) FILTER (WHERE market <> 'REG') AS other_trades
        FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE symbol IS NOT NULL GROUP BY 1,2
    ),
    screened AS (SELECT date, symbol FROM 'daily_stats_ALL.parquet')
    SELECT CASE WHEN reg_trades = 0 THEN 'NDM/other only' ELSE 'has REG trades' END AS kind,
           count(*) AS dropped_symbol_days
    FROM traded t LEFT JOIN screened s ON t.date=s.date AND t.symbol=s.symbol
    WHERE s.symbol IS NULL
    GROUP BY 1 ORDER BY 2 DESC
""").df().to_string(index=False))


          kind  dropped_symbol_days
has REG trades                 2357
NDM/other only                 1723


In [13]:

# B. How much notional is NDM contaminating the screen with?
print(duckdb.sql(f"""
    SELECT market, count(*) AS trades,
           round(sum(qty*price)/1e6, 1) AS notional_m,
           round(100.0*sum(qty*price) / sum(sum(qty*price)) OVER (), 2) AS pct_notional
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    GROUP BY market ORDER BY notional_m DESC
""").df().to_string(index=False))

       market   trades  notional_m  pct_notional
          REG 84424653   8746034.3         71.94
STOCK_DEL_FUT 14353938   2936023.3         24.15
          NDM    15090    464837.4          3.82
 STOCK_CS_FUT     6888     11017.1          0.09
      ODD_LOT     4670         9.4          0.00


In [16]:
print(duckdb.sql(f"""
    SELECT symbol,
           round(sum(qty*price) FILTER (WHERE market='NDM')/1e6, 1) AS ndm_m,
           round(sum(qty*price) FILTER (WHERE market='REG')/1e6, 1) AS reg_m,
           round(100.0*sum(qty*price) FILTER (WHERE market='NDM')
                 / nullif(sum(qty*price) FILTER (WHERE market='REG'),0), 2) AS ndm_pct_of_reg,
           count(*) FILTER (WHERE market='NDM') AS ndm_trades
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    WHERE symbol IN ('KTML','FFC','SYS','BAFL','PSO','HBL','KOHC','PKGS','AKBL','NBP','SGPL','OGDC')
    GROUP BY symbol ORDER BY ndm_pct_of_reg DESC NULLS LAST
""").df().to_string(index=False))

symbol   ndm_m    reg_m  ndm_pct_of_reg  ndm_trades
  AKBL  6224.9  58568.6           10.63          80
  KOHC  2307.3  25028.7            9.22         109
   PSO 20469.4 319352.5            6.41         360
  SGPL   661.5  11800.0            5.61         101
   HBL  6155.8 115724.1            5.32         139
   NBP 19793.6 390173.1            5.07         298
  KTML   262.1   9990.1            2.62          24
   FFC  6997.4 287117.5            2.44         153
  BAFL  1127.8  52817.3            2.14          44
  OGDC  6285.6 340186.5            1.85         189
   SYS  1182.4  91163.7            1.30          42
  PKGS    28.5   2474.4            1.15           6


In [17]:
print(duckdb.sql(f"""
    SELECT symbol,
           round(sum(qty*price) FILTER (WHERE market='NDM')/1e6, 1) AS ndm_m,
           round(sum(qty*price) FILTER (WHERE market='REG')/1e6, 1) AS reg_m,
           round(100.0*sum(qty*price) FILTER (WHERE market='NDM')
                 / nullif(sum(qty*price) FILTER (WHERE market='REG'),0), 2) AS ndm_pct_of_reg
    FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
    WHERE symbol IN ('KTML','FFC','SYS','BAFL','PSO','HBL','KOHC','PKGS','AKBL','NBP','SGPL','OGDC')
    GROUP BY symbol ORDER BY ndm_pct_of_reg DESC NULLS LAST
""").df().to_string(index=False))

symbol   ndm_m    reg_m  ndm_pct_of_reg
  AKBL  6224.9  58568.6           10.63
  KOHC  2307.3  25028.7            9.22
   PSO 20469.4 319352.5            6.41
  SGPL   661.5  11800.0            5.61
   HBL  6155.8 115724.1            5.32
   NBP 19793.6 390173.1            5.07
  KTML   262.1   9990.1            2.62
   FFC  6997.4 287117.5            2.44
  BAFL  1127.8  52817.3            2.14
  OGDC  6285.6 340186.5            1.85
   SYS  1182.4  91163.7            1.30
  PKGS    28.5   2474.4            1.15


In [19]:
import duckdb, pandas as pd, sys
sys.path.insert(0, "existing_mm")              # adjust if needed
from ticker_stats_core import stats_for_symbol

PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
TCOLS = "symbol,transact_time,initiator,price,qty,aggressor_side,market"
SCOLS = "symbol,msg_seq,orig_time,entry_type,level,px"
FEE   = "ceiling_pkr_rt_2p00"

rows = []
for SYM in ("AKBL", "KOHC", "PSO", "NBP"):
    dates = duckdb.sql(f"""
        SELECT DISTINCT date FROM read_parquet('{PARSED}/trades/*/*.parquet', hive_partitioning=true)
        WHERE symbol='{SYM}' AND market='NDM' ORDER BY date
    """).df()["date"].map(lambda x: str(x)[:10]).tolist()[:10]          # first 10 affected days per symbol

    for d in dates:
        t = duckdb.sql(f"SELECT {TCOLS} FROM read_parquet('{PARSED}/trades/date={d}/*.parquet') "
                       f"WHERE symbol='{SYM}'").df()
        s = duckdb.sql(f"SELECT {SCOLS} FROM read_parquet('{PARSED}/ob_snapshot/date={d}/*.parquet') "
                       f"WHERE symbol='{SYM}'").df()
        a = stats_for_symbol(s, t, SYM)                          # as computed now
        b = stats_for_symbol(s, t[t.market != "NDM"], SYM)        # NDM excluded
        if a and b:
            rows.append({"symbol": SYM, "date": d,
                         "ceiling_with_ndm": a[FEE], "ceiling_clean": b[FEE],
                         "inflation_x": (a[FEE] / b[FEE]) if b[FEE] > 0 else float("inf"),
                         "notional_with": a["notional_m"], "notional_clean": b["notional_m"]})

df = pd.DataFrame(rows)
pd.set_option("display.width", 220, "display.max_columns", None)
print(df.round(2).to_string(index=False))
print("\nmedian inflation per symbol (affected days only):")
print(df.groupby("symbol")["inflation_x"].median().round(3).to_string())

symbol       date  ceiling_with_ndm  ceiling_clean  inflation_x  notional_with  notional_clean
  AKBL 2025-09-01         632523.08      632523.08         1.00         581.53          581.16
  AKBL 2025-09-03         182878.79      182855.23         1.00         342.40          342.36
  AKBL 2025-09-12         223748.79      223014.39         1.00         176.86          176.13
  AKBL 2025-09-17          86283.17       84608.87         1.02         132.16          123.90
  AKBL 2025-09-24         208928.55      208928.55         1.00         163.39          162.57
  AKBL 2025-09-30         448367.35      441547.28         1.02         454.62          441.07
  AKBL 2025-10-03         225190.99      224603.38         1.00         246.97          243.72
  AKBL 2025-10-07         521783.08      521476.55         1.00         545.91          545.50
  AKBL 2025-10-09         222520.57      222520.57         1.00        1112.67          216.37
  AKBL 2025-10-14         785148.75      785148.75

In [1]:
import duckdb
ds = duckdb.sql("SELECT * FROM 'daily_stats_enriched.parquet'").df()
reg = ds[(ds.segment=="REG") & (ds.regime=="normal")]
med = reg.groupby("symbol")["ceiling_paired_rt_2p00"].median()
for thr in (0, 1_000, 5_000, 10_000, 20_000, 50_000):
    print(f"  MATERIAL_PKR={thr:>7,} -> {int((med>thr).sum()):3d} REG symbols clear it")
print(f"\n  median REG normal-day ceiling: {med.median():,.0f}")
print(f"  p90: {med.quantile(0.9):,.0f}   p99: {med.quantile(0.99):,.0f}")

  MATERIAL_PKR=      0 -> 515 REG symbols clear it
  MATERIAL_PKR=  1,000 -> 392 REG symbols clear it
  MATERIAL_PKR=  5,000 -> 279 REG symbols clear it
  MATERIAL_PKR= 10,000 -> 215 REG symbols clear it
  MATERIAL_PKR= 20,000 -> 152 REG symbols clear it
  MATERIAL_PKR= 50,000 ->  84 REG symbols clear it

  median REG normal-day ceiling: 6,641
  p90: 82,575   p99: 312,119


In [2]:
med35 = ds[(ds.segment=="REG") & (ds.regime=="normal")].groupby("symbol")["ceiling_paired_rt_35p45"].median()
for thr in (0, 1_000, 5_000, 10_000, 20_000, 50_000):
    print(f"  35bps MATERIAL_PKR={thr:>7,} -> {int((med35>thr).sum()):3d} REG symbols clear it")
print(f"  median: {med35.median():,.0f}  p90: {med35.quantile(0.9):,.0f}")

  35bps MATERIAL_PKR=      0 -> 514 REG symbols clear it
  35bps MATERIAL_PKR=  1,000 -> 347 REG symbols clear it
  35bps MATERIAL_PKR=  5,000 -> 146 REG symbols clear it
  35bps MATERIAL_PKR= 10,000 ->  57 REG symbols clear it
  35bps MATERIAL_PKR= 20,000 ->  16 REG symbols clear it
  35bps MATERIAL_PKR= 50,000 ->   5 REG symbols clear it
  median: 2,149  p90: 10,365


In [4]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", None)
p2 = pd.read_csv("persistence_REG_2p00.csv")

print("=== top 25 at 2bps by pct_days_top20 ===")
print(p2.head(25)[["symbol","pct_days_top20","pct_days_top20_rankonly",
                   "pct_days_unscreenable","ceiling_median","iqr_over_median",
                   "notional_m_median","spread_bps_median","days_traded"]].round(3).to_string(index=False))

=== top 25 at 2bps by pct_days_top20 ===
symbol  pct_days_top20  pct_days_top20_rankonly  pct_days_unscreenable  ceiling_median  iqr_over_median  notional_m_median  spread_bps_median  days_traded
   PPL           0.773                    0.773                    0.0      471958.987            0.955           1439.333              7.352          207
   NBP           0.763                    0.763                    0.0      508306.233            1.263           1368.288              8.707          207
  OGDC           0.580                    0.580                    0.0      320496.897            0.881           1378.049              5.703          207
   PSO           0.517                    0.517                    0.0      333951.603            0.989           1193.638              6.572          207
   BOP           0.507                    0.507                    0.0      314489.301            1.525           1293.462              5.753          207
   PTC           0.469       

In [6]:
import duckdb
duckdb.sql("""
    SELECT count(*)                              AS n_rows,
           count(DISTINCT date)                  AS n_dates,
           count(net60_bps)                      AS has_net60,
           round(avg(net60_bps), 2)              AS avg_net60,
           round(100.0 * avg(CASE WHEN net60_bps > 0 THEN 1 ELSE 0 END), 1) AS pct_viable
    FROM 'daily_stats_ALL.parquet'
""").df()

,n_rows,n_dates,has_net60,avg_net60,pct_viable
0,122880,207,111351,15.34,32.8


In [7]:
import duckdb
duckdb.sql("""
    SELECT
      round(avg(net60_bps), 2)                                   AS simple_avg,
      round(sum(net60_bps * notional_m) / sum(notional_m), 2)    AS notional_wtd_avg,
      round(avg(net60_bps) FILTER (WHERE notional_m > 100), 2)   AS avg_liquid_only,
      round(100.0*avg(CASE WHEN net60_bps>0 THEN 1 ELSE 0 END)
            FILTER (WHERE notional_m > 100), 1)                  AS pct_viable_liquid
    FROM 'daily_stats_ALL.parquet'
    WHERE net60_bps IS NOT NULL
""").df()

,simple_avg,notional_wtd_avg,avg_liquid_only,pct_viable_liquid
0,15.34,-15.95,-15.75,1.5


In [ ]:
What this means, stated plainly: on the liquid PSX names, at the retail fee, a naive passive market maker loses money after adverse selection. Only 1.5% of liquid symbol-days survive. The +15bps you were about to celebrate was an artifact of counting a wide quoted spread on a symbol that traded five times as if it were a capturable edge. It isn't.

In [8]:
import duckdb
# net60 at retail = mk60 - 17.73. Add back 16.1bps to approximate the ~1.6bps TREC RT/2 side.
duckdb.sql("""
    SELECT
      round(avg(mk60_bps) FILTER (WHERE notional_m>100), 2)                        AS gross_mk60_liquid,
      round(avg(mk60_bps - 0.16*100) FILTER (WHERE notional_m>100), 2)             AS net60_at_trec,
      round(100.0*avg(CASE WHEN (mk60_bps - 0.16*100)>0 THEN 1 ELSE 0 END)
            FILTER (WHERE notional_m>100), 1)                                      AS pct_viable_trec
    FROM 'daily_stats_ALL.parquet' WHERE mk60_bps IS NOT NULL
""").df()

,gross_mk60_liquid,net60_at_trec,pct_viable_trec
0,1.98,-14.02,1.9


Markdown